In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from scipy import interpolate
from os.path import join
import sys
import pandas as pd

repo_root_path = join('..','..')
python_modules_path = join(repo_root_path, 'src', 'python')
if python_modules_path not in sys.path:
    sys.path.append(python_modules_path)

from specie_properties.common import AW, MW
from scm.scm import ilmenite, ilmenite_orig, concentration_from_molar_fraction, Abad_SCR

mm = 1.0/2.54/10

plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))
path_wpd = os.path.join('validation')

In [ ]:
T = 950+273.15
# C = concentration_from_molar_fraction(T=T, x=0.15)
C = concentration_from_molar_fraction(T=T, x=0.21)
r = ilmenite_orig['pre']['O2']
tau = r.tau(T=T, C=C)
t = np.array(np.arange(0,tau,1))
Xs = r.conversion(T, C, t)
plt.plot(t, Xs, 'g--', label='Analytical')
plt.axhline(0.25, linestyle='--')

In [ ]:
print(Abad_SCR.conversion(r,T,C,71))
# r.conversion(T,C,0)
print(Abad_SCR.tau(r,T,C))
print(Abad_SCR._conversion(r, 1))

In [ ]:
class zeroDimensionalCase:
    g_species = ("N2", "O2", "CO", "CO2", "H2O", "H2", "CH4")#, "Ar")
    """List of gaseous species"""

    s_species = ("Fe2O3", "FeO", "inerts")
    """List of solid species"""

    reactions = ['Abad_' + sp for sp in ('CH4','CO','H2','O2')]
    """List of reactions"""

    def __init__(self, root_path, folder_name) -> None:
        self.case_specie  = folder_name.split('_')[-1]
        self.case_carrier = folder_name.split('_')[-2]

        pp_path = os.path.join(folder_name,"postProcessing")
        path_probes = os.path.join(pp_path,"probesFn","0","{}.{}")
        path_s      = os.path.join(pp_path,"volIntegrateFn(fields=(rho.solids),weightField=alpha.solids)","0","volFieldValue.dat")
        path_g      = os.path.join(pp_path,"volIntegrateFn(fields=(rho.gas),weightField=alpha.gas)","0","volFieldValue.dat")
        path_g_sp   = os.path.join(pp_path,'volIntegrateFn(fields=({}.gas),weightFields=(alpha.gasrho.gas))',"0","volFieldValue.dat")
        path_s_sp   = os.path.join(pp_path,'volIntegrateFn(fields=({}.solids),weightFields=(alpha.solidsrho.solids))',"0","volFieldValue.dat")
        path_NRR    = os.path.join(pp_path,"volIntegrateFn(fields=(NRR_{}))","0","volFieldValue.dat")

        self.t = np.loadtxt(path_s, unpack=True)[0]
        """Time [s]"""

        self.dat_s = np.loadtxt(path_s, unpack=True)[1]
        """Mass of solids [kg]"""
        
        self.dat_g = np.loadtxt(path_g, unpack=True)[1]
        """Mass of gases [kg]"""

        self.dat_NRRs = {reaction: np.loadtxt(path_NRR.format(reaction), unpack=True)[1] for reaction in self.reactions}
        """Dictionary of reaction rates vs time [mol/s]"""

        self.dat_g_sp = {g_sp: np.loadtxt(path_g_sp.format(g_sp), unpack=True)[1] for g_sp in self.g_species}
        """Dictionary of total mass of gaseous species [kg]"""

        self.dat_s_sp = {s_sp: np.loadtxt(path_s_sp.format(s_sp), unpack=True)[1] for s_sp in self.s_species}
        """Dictionary of total mass of solid species [kg]"""
        
        self.percent_O = self.dat_s_sp['Fe2O3']/MW('Fe2O3')*AW('O')/self.dat_s[-1]
        """Percent of oxigen available for reactions"""
        
        self.Ro = max(self.percent_O)
        """Oxygen carrying capacity"""

        self.conversion = self.percent_O/self.Ro if self.case_specie == 'O2' else 1 - self.percent_O/self.Ro
        """Relative conversion"""
        
        # dat_s_sp = dict()
        # dat_Y_s_sp = dict()
        # for sp in s_species:
        #     dat_s_sp[sp] = np.loadtxt(path_s_sp.format(sp), unpack=True) # [kg]
        #     dat_Y_s_sp[sp] = np.loadtxt(path_probes.format(sp, 'solids'), unpack=True) # [kg]


In [ ]:
# TODO: Integrate NRR

In [ ]:
def fix_wpd_col_names(dfcols):
    # Fix column names
    newcolumns = []
    for i, c in enumerate(dfcols):
        if 'Unnamed' in c[0]:
            newcolumns.append( (dfcols[i-1][0], dfcols[i][1]) )
        else:
            newcolumns.append( c )
    dfcols = pd.MultiIndex.from_tuples(newcolumns)
    return dfcols

from cycler import cycler
# cc = (cycler(linestyle=['none'])*cycler(fillstyle=dict('none'))*cycler(marker=['o', '^', 'D', 's', ]))
cc = (cycler(linestyle=['none'],mfc=['none'])*cycler(marker=['o', '^', 'D', 's', ]))
for d in cc:
    print(d)
plt.rc('axes', prop_cycle=cc)

In [ ]:
# path_wpd = os.path.join('..', '..', '..', 'web_plot_digitizer')

def plot_results_for_fig(fig, axs, fign, fuel, vardim, concsim, Tsim=1173, carriers = ('pre', 'act')):
    exp_path = os.path.join(path_wpd, f'Abad2011_fig{fign:02}.csv')
    df = pd.read_csv(exp_path, header=[0,1])
    df.columns = fix_wpd_col_names(df.columns)
    vararr = sorted(list(set([int(el[0].split('_')[1]) for el in df.columns.to_list()]))) # varied parameters

    for ax, carrier in zip(axs, carriers):
        for conc in vararr:
            ax.plot(df[f'{carrier}_{conc}']['X'], df[f'{carrier}_{conc}']['Y'], label=f'{conc} {vardim}')
        ax.set(
            xlabel='time (s)',
            ylabel='Conversion (-)',
            ylim=(0,1)
        )

        case_folder = f'0D_{carrier}_{fuel}'
        case = zeroDimensionalCase('.', case_folder)
        ax.plot(case.t, case.conversion, 'r-', label='Simulated')

        ###
        try: 
            C = concentration_from_molar_fraction(Tsim, concsim)
            r = ilmenite[carrier][fuel]
            tau = r.tau(Tsim, C)
            t = np.array(np.arange(0,tau,1))
            Xs = r.conversion(Tsim, C, t)
            ax.plot(t, Xs, 'g--', label='Analytical')
        except NotImplementedError:
            print(f'Analytical solution not implemented for {carrier} {fuel}')
            pass
        ###

        ax.legend()
        fig.suptitle(f'Figure {fign}')

In [ ]:
fig, axs = plt.subplots(figsize = (9, 4), ncols = 2)
plot_results_for_fig(
    fig=fig,
    axs=axs,
    fign = 6,
    fuel = 'H2',
    vardim = 'vol%',
    Tsim = 1173,
    concsim = 0.15,
    carriers = ('pre', 'act'),
)
axs[0].set(
    xlim=(0,900)
)
axs[1].set(
    xlim=(0,180)
)

In [ ]:
fig, axs = plt.subplots(figsize = (9, 4), ncols = 2)
plot_results_for_fig(
    fig=fig,
    axs=axs,
    fign = 7,
    fuel = 'CO',
    vardim = 'K',
    Tsim = 1173,
    concsim = 0.15,
    carriers = ('pre', 'act'),
)
axs[0].set(
    xlim=(0,900)
)
axs[1].set(
    xlim=(0,180)
)

In [ ]:
fig, axs = plt.subplots(figsize = (9, 4), ncols = 2)
plot_results_for_fig(
    fig=fig,
    axs=axs,
    fign = 9,
    fuel = 'O2',
    vardim = 'vol%',
    Tsim = 1173,
    concsim = 0.21,
    carriers = ('pre', 'act'),
)
axs[0].set(
    xlim=(0,900)
)
axs[1].set(
    xlim=(0,60)
)

## Figure for the paper

In [ ]:
plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))

for carrier in ('pre', 'act'):
    fig, ax = plt.subplots(figsize=[90*mm, 67.5*mm])
    for fuel, label in {'CH4':r'$\mathrm{CH}_4$', 'CO': r'$\mathrm{CO}$', 'H2':r'$\mathrm{H}_2$', 'O2': r'$\mathrm{O}_2$'}.items():
        t, X = np.loadtxt(os.path.join(path_wpd, f'Abad2011_fig5_{carrier}_{fuel}.dat'), unpack=True)
        h, = ax.plot(t, 1-X if fuel == 'O2' else X, 'x', label=label)

        case_folder = f'0D_{carrier}_{fuel}'
        case = zeroDimensionalCase('.', case_folder)
        h, = ax.plot(case.t, 1-case.conversion  if fuel == 'O2' else case.conversion, '-', color=h.get_color())

        try:
            Tsim = 1173.15 
            C = concentration_from_molar_fraction(Tsim, 0.21 if fuel == 'O2' else 0.15)
            r = ilmenite[carrier][fuel]
            tau = r.tau(Tsim, C)
            t = np.array(np.arange(0,tau,1))
            Xs = r.conversion(Tsim, C, t)
            ax.plot(t, 1-Xs if fuel == 'O2' else Xs, 'k', linestyle=(0, (5, 10)))
        except NotImplementedError:
            pass
        
    ax.set(
        xlabel = r'$t$, s',
        ylabel = r'$X, -$',
    )
    ax.legend()
    # fig.savefig(f'plots/Abad2011_fig5_{carrier}.pdf', bbox_inches='tight')

# Reaction heat and temperature

In [ ]:
path_p  = os.path.join('0D_act_O2',"postProcessing","probesFn","0","p")
dat_p   = np.loadtxt(path_p, unpack=True)
dat_p

In [ ]:
root_path = '.'
carrier = 'act'
reaction = 'O2'

case_path = '_'.join(['0D', carrier, reaction])
path_Hs = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.solids),weightFields=(alpha.solidsrho.solids))","0","volFieldValue.dat")
path_Hg = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.gas),weightFields=(alpha.gasrho.gas))","0","volFieldValue.dat")
path_p  = os.path.join(root_path,case_path,"postProcessing","probesFn","0","p")
dat_Hs = np.loadtxt(path_Hs, unpack=True)
dat_Hg = np.loadtxt(path_Hg, unpack=True)
dat_p   = np.loadtxt(path_p, unpack=True)

v_domain = 1
alpha_s = 1e-6
change_g = dat_Hg[1][-1]-dat_Hg[1][0] - v_domain*(dat_p[1][-1]-dat_p[1][0])
change_s = dat_Hs[1][-1]-dat_Hs[1][0] - alpha_s*v_domain*(dat_p[1][-1]-dat_p[1][0])
dat_Hg[1][-1]+dat_Hs[1][-1] #-dat_Hg[1][0]

In [ ]:
# See also PP.py

carriers  = ['act', 'pre']
# carriers  = ['act']
reactions = ['CH4', 'CO', 'H2', 'O2']

cases = [
    '0D_act_CH4',
    '0D_act_CO',
    '0D_act_H2',
    '0D_act_O2'
]

m_ilm = 0.000050 # [kg] - mass of ilmenite (see solidsMass in caseParamsDict)
v_domain = 1 # see L in caseParamsDict

root_path = '.'

W_O = 15.9994e-3

def calc_reaction_heat_per_O_atom(carrier, reaction, case_path = None):
    if case_path is None:
            case_path = '_'.join(['0D', carrier, reaction])
    if carrier == 'act': # see caseParamsDict
        Ro = 0.033
        rhos = 4100
    else:
        Ro = 0.040
        rhos = 4250
    
    rhomO = Ro/W_O # [mol/kg solid] - molar density of reacting O atoms in solid phase
    path_Hs = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.solids),weightFields=(alpha.solidsrho.solids))","0","volFieldValue.dat")
    path_Hg = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.gas),weightFields=(alpha.gasrho.gas))","0","volFieldValue.dat")
    path_p  = os.path.join(root_path,case_path,"postProcessing","probesFn","0","p")

    dat_Hs = np.loadtxt(path_Hs, unpack=True)
    dat_Hg = np.loadtxt(path_Hg, unpack=True)
    dat_p  = np.loadtxt(path_p, unpack=True)

    alpha_s = m_ilm/v_domain/rhos # calculated in caseDict as "$solidsMass / $domainVolume / $rho_solids"
    change_g = dat_Hg[1][-1]-dat_Hg[1][0] - (1-alpha_s)*v_domain*(dat_p[1][-1]-dat_p[1][0])
    change_s = dat_Hs[1][-1]-dat_Hs[1][0] - alpha_s*v_domain*(dat_p[1][-1]-dat_p[1][0])

    change_total = change_g+change_s

    m = m_ilm if reaction != 'O2' else m_ilm/(1-Ro)
    # print(f"{case_path:10s}: {change_total/m:10.2f} J/kg oxidized ilmenite")
    # print(f"{case_path:10s}: {change_total/m/rhomO/1000:10.2f} kJ/mol O atom")
    return change_total/m/rhomO

print(f'Released heat per O atom at T={900}°C')
for carrier in carriers:
    for reaction in reactions:
        case_path = '_'.join(['0D', carrier, reaction])
        Q = calc_reaction_heat_per_O_atom(carrier, reaction)
        print(f"{case_path:10s}: {Q/1000:10.2f} kJ/mol O atom")


In [ ]:
m_sample = 0.000050 # [kg] - sample mass
Ro = 0.033
nO = m_sample*Ro/MW('O')

# root_paths = ['.', 'BACKUP_massSourceOld']
root_paths = ['.']
case_path = '0D_act_O2'
fig, ax = plt.subplots()
for root_path in root_paths:
    path_hgas = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.gas),weightFields=(alpha.gasrho.gas))","0","volFieldValue.dat")
    path_hsolids = os.path.join(root_path,case_path,"postProcessing","volIntegrateFn(fields=(h.solids),weightFields=(alpha.solidsrho.solids))","0","volFieldValue.dat")
    dat_hgas = np.loadtxt(path_hgas, unpack=True)
    dat_hsolids = np.loadtxt(path_hsolids, unpack=True)
    ax.plot(dat_hgas[0], dat_hgas[1]+dat_hsolids[1])
    # plt.plot(dat_hgas[0], dat_hgas[1])
    dH = (dat_hgas[1][-1] + dat_hsolids[1][-1]) - (dat_hgas[1][0] + dat_hsolids[1][0])
    # print(dH, 'J, ', dH/m_sample/1000, 'kJ/kg')
    
    print(dH, 'J, ', dH/m_sample/1000, 'kJ/kg')
ax.legend(['new', 'old'])
ax.set(xlabel=r'$t, \mathrm{s}$',ylabel=r'$H_\mathrm{tot}, \mathrm{J}$')

In [ ]:
root_path = '.'
case_path = '0D_act_CH4'
path_Tgas = os.path.join(root_path,case_path,"postProcessing","probesFn","0","T.gas")
dat_Tgas = np.loadtxt(path_Tgas, unpack=True)
fig, ax = plt.subplots()
ax.plot(dat_Tgas[0], dat_Tgas[1], label='massSource new')

# root_path = 'BACKUP_massSourceOld'
# # case_path = '0D_act_O2'
# path_Tgas = os.path.join(root_path,case_path,"postProcessing","probesFn","0","T.gas")
# dat_Tgas = np.loadtxt(path_Tgas, unpack=True)
# ax.plot(dat_Tgas[0], dat_Tgas[1], label='massSource old')

ax.legend()